# Vígil.ia — RT-DETR (detecção) · treino em 3 etapas

Plano:
1. **Base** — RT-DETR nas 12.528 imagens do dataset original.
2. **Fine-tune** — nas suas fotos reais (celular).
3. **Teste em vídeo** — ver onde erra → justifica a etapa de treino em vídeo (anotação real).

### O pulo do gato: pseudo-rótulo
O dataset original é de **classificação** (pastas por classe, 1 grão, **sem caixas**).
RT-DETR é **detector** — exige caixas. Então a gente **gera as caixas automaticamente**
com **Otsu** (mesmo algoritmo do `crop_single_grain`): cada imagem tem 1 grão em fundo
escuro → Otsu acha o grão → vira 1 caixa, e a classe vem da pasta. Zero anotação manual.

**Caveat honesto:** esse base aprende a ver **um grão isolado**, não vários amontoados.
O teste em vídeo (etapa 3) provavelmente ainda vai falhar em cenas densas — e é **isso
que queremos observar**, é o que motiva a anotação de vídeo depois. O `mosaic` (cola 4
imagens numa) ajuda a simular multi-grão de graça durante o treino.

> Rodar com GPU (A100 ou melhor). O YOLO11s-cls atual continua intacto pro modo foto.

## 1. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU! Troque o ambiente de execução.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Caminhos dos dados
Ajuste os caminhos do seu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Dataset de CLASSIFICAÇÃO (12.5k). Pode apontar pra pasta "de cima" — o builder
# procura train/valid/test em qualquer profundidade lá dentro.
CLS_BASE = '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder'

# Suas fotos reais (fine-tune): pastas com nomes PT das classes (busca recursiva)
REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]

# Vídeo de teste (etapa 3): grave um vídeo curto com VÁRIOS grãos, fundo escuro
VIDEO_TESTE = '/content/drive/MyDrive/teste_soja.mp4'

In [ ]:
# Diagnóstico: mostra a estrutura real do CLS_BASE (2 níveis) pra conferir
# onde estão train/valid/test antes de construir o dataset.
import os
assert os.path.isdir(CLS_BASE), f'CLS_BASE não existe: {CLS_BASE}'
for root, dirs, files in sorted(os.walk(CLS_BASE)):
    depth = root[len(CLS_BASE):].count(os.sep)
    if depth > 2:
        dirs[:] = []
        continue
    print('  ' * depth + (os.path.basename(root) or CLS_BASE) + f'/  ({len(files)} arquivos)')

## 3. Pseudo-rotulador (Otsu → caixa) + builder do dataset de detecção
Transforma as pastas de classificação num dataset YOLO-detect (imagens + labels + data.yaml).
As imagens são **regravadas localmente** (não symlink) — treinar lendo do Drive é lento demais.

In [ ]:
import glob, hashlib, unicodedata, cv2, yaml

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
SPLIT_MAP = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def letterbox640(img, size=640):
    """Padroniza TODAS as imagens em 640x640 (resize + borda preta centrada).
    O RT-DETR nao redimensiona na validacao; fotos de celular tem tamanhos
    variados e quebrariam o batch. Devolve tambem escala/offsets p/ mapear a caixa."""
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def build_detection_set(items, out_dir):
    assert items, ('Nenhuma imagem coletada! Rode a célula de diagnóstico acima e '
                   'confira o caminho/estrutura das pastas.')
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    kept = skipped = 0
    per = {}
    for i, (path, cls, sp) in enumerate(items):
        if i % 1000 == 0:
            print(f'  {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        # Otsu na imagem ORIGINAL: com a borda preta aplicada antes, um fundo
        # cinza (fotos de celular) vira "foreground" e a caixa sai do tamanho
        # da foto inteira -> mAP 0 no fine-tune.
        box = otsu_box(img)
        if box is None:
            skipped += 1; continue
        img, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        stem = f'{sp}_{i:06d}'
        # regrava LOCAL (jpg): treino lê do disco da máquina, não do Drive
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', img,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        with open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w') as f:
            f.write(f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}')
        kept += 1
        per[(sp, cls)] = per.get((sp, cls), 0) + 1
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    print(f'kept={kept}  skipped={skipped}')
    for sp in ('train', 'val', 'test'):
        row = {NAMES[c]: per.get((sp, c), 0) for c in range(5)}
        if sum(row.values()):
            print(sp, row)
    assert kept > 0, 'Nenhuma caixa gerada — Otsu falhou em tudo? Confira as imagens.'
    return f'{out_dir}/data.yaml'

def collect_base(base_dir):
    """Acha train/valid/test em QUALQUER profundidade dentro de base_dir
    (o unzip costuma criar uma pasta extra no meio)."""
    items = []
    for root, dirs, _ in os.walk(base_dir):
        for d in list(dirs):
            sp = SPLIT_MAP.get(d.lower())
            if sp is None:
                continue
            split_dir = os.path.join(root, d)
            for folder in sorted(os.listdir(split_dir)):
                cls = class_of(folder)
                if cls is None:
                    continue
                for p in glob.glob(os.path.join(split_dir, folder, '*')):
                    if p.lower().endswith(IMG_EXT):
                        items.append((p, cls, sp))
            dirs.remove(d)  # não desce dentro do split de novo
    from collections import Counter
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    return items

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    sp = 'val' if (h % 100) < val_frac * 100 else 'train'
                    items.append((p, cls, sp))
    from collections import Counter
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    return items

In [ ]:
# Constrói o dataset base (12.5k) com caixas pseudo-rotuladas.
# Lê tudo do Drive UMA vez (uns 10–25 min) e regrava local — depois o treino voa.
BASE_YAML = build_detection_set(collect_base(CLS_BASE), '/content/soja_det_base')
print('data.yaml ->', BASE_YAML)

In [ ]:
# Sanidade visual: confere se as caixas do Otsu estão certas ANTES de treinar
import matplotlib.pyplot as plt
sample = glob.glob('/content/soja_det_base/images/train/*.jpg')[:6]
plt.figure(figsize=(12, 7))
for i, p in enumerate(sample):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    parts = open(lbl).read().split()
    cls = int(parts[0])
    cx, cy, ww, hh = [float(x) for x in parts[1:]]
    x1, y1 = int((cx - ww / 2) * w), int((cy - hh / 2) * h)
    x2, y2 = int((cx + ww / 2) * w), int((cy + hh / 2) * h)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    ax = plt.subplot(2, 3, i + 1); ax.imshow(img); ax.set_title(NAMES[cls]); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Etapa 1 — RT-DETR base (12.5k)
`rtdetr-l` pré-treinado COCO → transfer learning. `mosaic` simula multi-grão.

In [ ]:
from ultralytics import RTDETR

BATCH = 16   # A100 40GB. Com 80GB+ de VRAM pode subir p/ 32-48. Se der OOM, baixe.

model = RTDETR('rtdetr-l.pt')
model.train(
    data=BASE_YAML, epochs=50, imgsz=640, batch=BATCH, device=0, seed=42,
    optimizer='AdamW', lr0=1e-4, patience=20, close_mosaic=10, amp=True,
    cache=False, workers=8,
    mosaic=1.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
    degrees=15, translate=0.1, scale=0.5, fliplr=0.5, flipud=0.5,
    project='runs_rtdetr', name='base_12k', exist_ok=True,
)

In [ ]:
# mAP do base no split de teste
r = RTDETR('runs_rtdetr/base_12k/weights/best.pt').val(
    data=BASE_YAML, split='test', imgsz=640, device=0)
print(f'BASE  mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}')

## 5. Etapa 2 — Fine-tune nas suas fotos reais
lr menor (não esquecer o que aprendeu no base). Mesmo pseudo-rótulo Otsu.

In [ ]:
REAL_YAML = build_detection_set(collect_real(REAL_SRCS), '/content/soja_det_real')

ft = RTDETR('runs_rtdetr/base_12k/weights/best.pt')
ft.train(
    data=REAL_YAML, epochs=60, imgsz=640, batch=BATCH, device=0, seed=42,
    optimizer='AdamW', lr0=5e-5, patience=25, close_mosaic=10, amp=True,
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_rtdetr', name='ft_real', exist_ok=True,
)

## 6. Etapa 3 — Teste em vídeo (o momento da verdade)
Roda no vídeo real e salva anotado. É aqui que se vê se precisa (e quanto) do treino em vídeo.

In [ ]:
best = RTDETR('runs_rtdetr/ft_real/weights/best.pt')
best.predict(source=VIDEO_TESTE, conf=0.25, save=True,
             project='runs_rtdetr', name='pred_video', exist_ok=True)
print('Vídeo anotado -> runs_rtdetr/pred_video/')

# QC premium no 1º frame: % intacto >= 90% -> Aprovado
frame = next(iter(best.predict(source=VIDEO_TESTE, conf=0.25, stream=True)))
ids = frame.boxes.cls.cpu().numpy().astype(int)
total = len(ids)
premium = int((ids == NAMES.index('intact')).sum())
pct = premium / total if total else 0
print(f'{total} graos | {premium} intactos ({pct*100:.0f}%) -> ' +
      ('APROVADO' if pct >= 0.9 else 'REPROVADO'))

## 7. Exportar
`.pt` pro servidor (HF Space) e `.onnx` caso a gente teste no browser depois.

In [ ]:
import shutil
shutil.copy('runs_rtdetr/ft_real/weights/best.pt', 'soja_rtdetr_finetuned.pt')
RTDETR('soja_rtdetr_finetuned.pt').export(format='onnx', imgsz=640, opset=16, simplify=True)
print('Gerado: soja_rtdetr_finetuned.pt (+ .onnx)')

from google.colab import files
files.download('soja_rtdetr_finetuned.pt')